In [1]:
import sys
import os
import tempfile
from autogen.agentchat import ConversableAgent

sys.path.append(os.path.abspath(".."))

from llm_config import llm_config

In [ ]:
from typing import Annotated, Literal

Operator = Literal["+", "-", "*", "/"]

def calculator(a:int, b:int, operator: Annotated[Operator, "operator"]) -> int: #Annotated allows only those values to be passed, anything else fails
    if operator == "+":
        return a+b
    
    elif operator == "-":
        return a-b
    
    elif operator == "*":
        return a*b

    elif operator == "/":
        return int(a/b)

    else:
        raise ValueError("Invalid Operator")

In [ ]:
assistant = ConversableAgent(
    name = "Assistant",
    system_message = """You are a helpful AI assistant. You can help with simple calculations.
                        Return 'TERMINATE' when the task is done."""
    llm_config = llm_config
)

In [ ]:
#The user proxy agent is used to interact with the assistant agent and execute tool calls
user_proxy = ConversableAgent(
    name  = "User",
    llm_config = False,
    is_termination_msg = lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"],
    human_input_mode = "NEVER"
)

In [ ]:
#Register the tool signature with the assistant agent.
assistant.register_for_llm(name = 'calculator', description = "A simple calculator")(calculator)

#Register the tool function with the user proxy agent.
user_proxy.register_for_execution(name ="calculator")(calculator)

In [ ]:
from autogen import register_function

#Register the calculator function to the two agents

register_function(
    calculator,
    caller = assistant, #The assistant agent can suggest calls to the calculator
    executor = user_proxy, #The user proxy agent can execute the calculator calls
    name = "calculator", # by default the function name is used as the tool name
    description = "A simple calculator", # a description of the tool
)

In [ ]:
chat_result = user_proxy.initiate_chat(
    assistant, 
    message = "What is (44232 + 13312 / (232 -32)) * 5 ?"
    )

In [3]:
from autogen.agentchat import register_function


In [ ]:
assistant.llm_config["tools"]